In [40]:

!pip install -q datasets seqeval


In [41]:

from datasets import load_dataset

dataset = load_dataset("eriktks/conll2003")

print("Dataset loaded successfully!")
print(dataset)

print("\nAvailable splits:")
for split in dataset:
    print(f"{split}: {len(dataset[split])} sentences")

print("\nSample sentence:")
sample = dataset["train"][0]

print("Tokens:", sample["tokens"])
print("NER Tags:", sample["ner_tags"])


Using the latest cached version of the dataset since eriktks/conll2003 couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'conll2003' at /root/.cache/huggingface/datasets/eriktks___conll2003/conll2003/0.0.0/08e60c1beec5331740f0b1d3e2fbc2de7c4fcb49 (last modified on Tue Nov  3 21:29:35 2026).


Dataset loaded successfully!
DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

Available splits:
train: 14041 sentences
validation: 3250 sentences
test: 3453 sentences

Sample sentence:
Tokens: ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
NER Tags: [3, 0, 7, 0, 0, 0, 7, 0, 0]


In [42]:
from collections import Counter

# Get the actual label names from the dataset
label_names = dataset["train"].features["ner_tags"].feature.names

print("NER Label Mapping:")
for i, label in enumerate(label_names):
    print(f"{i}: {label}")

print("\nLabel Distribution:")
for split in ["train", "validation", "test"]:
    all_tags = [
        tag
        for sentence in dataset[split]["ner_tags"]
        for tag in sentence
    ]

    counts = Counter(all_tags)

    print(f"\n{split.upper()}:")
    for i, label in enumerate(label_names):
        print(f"{label}: {counts[i]}")

NER Label Mapping:
0: O
1: B-PER
2: I-PER
3: B-ORG
4: I-ORG
5: B-LOC
6: I-LOC
7: B-MISC
8: I-MISC

Label Distribution:

TRAIN:
O: 169578
B-PER: 6600
I-PER: 4528
B-ORG: 6321
I-ORG: 3704
B-LOC: 7140
I-LOC: 1157
B-MISC: 3438
I-MISC: 1155

VALIDATION:
O: 42759
B-PER: 1842
I-PER: 1307
B-ORG: 1341
I-ORG: 751
B-LOC: 1837
I-LOC: 257
B-MISC: 922
I-MISC: 346

TEST:
O: 38323
B-PER: 1617
I-PER: 1156
B-ORG: 1661
I-ORG: 835
B-LOC: 1668
I-LOC: 257
B-MISC: 702
I-MISC: 216


In [43]:
# Get label names from the dataset features
label_names = dataset["train"].features["ner_tags"].feature.names

print("Label Mapping:")
for i, label in enumerate(label_names):
    print(f"{i}: {label}")

# Verify token-label alignment
for split in ["train", "validation", "test"]:
    mismatches = sum(
        len(example["tokens"]) != len(example["ner_tags"])
        for example in dataset[split]
    )

    print(f"{split}: {mismatches} alignment errors")

# Display one sentence with its labels
sample = dataset["train"][0]

print("\nSample with NER Labels:")
for token, tag_id in zip(sample["tokens"], sample["ner_tags"]):
    print(f"{token:<15} {label_names[tag_id]}")

Label Mapping:
0: O
1: B-PER
2: I-PER
3: B-ORG
4: I-ORG
5: B-LOC
6: I-LOC
7: B-MISC
8: I-MISC
train: 0 alignment errors
validation: 0 alignment errors
test: 0 alignment errors

Sample with NER Labels:
EU              B-ORG
rejects         O
German          B-MISC
call            O
to              O
boycott         O
British         B-MISC
lamb            O
.               O


In [44]:
from collections import Counter

# Special tokens
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

# Build word vocabulary from training data only
word_counter = Counter(
    token.lower()
    for example in dataset["train"]
    for token in example["tokens"]
)

word_vocab = {
    PAD_TOKEN: 0,
    UNK_TOKEN: 1
}

for word in word_counter:
    word_vocab[word] = len(word_vocab)

# Build character vocabulary from training data only
char_counter = Counter(
    char
    for example in dataset["train"]
    for token in example["tokens"]
    for char in token
)

char_vocab = {
    PAD_TOKEN: 0,
    UNK_TOKEN: 1
}

for char in char_counter:
    char_vocab[char] = len(char_vocab)

# Label mappings
label2id = {
    label: idx
    for idx, label in enumerate(label_names)
}

id2label = {
    idx: label
    for label, idx in label2id.items()
}

print(f"Word vocabulary size: {len(word_vocab):,}")
print(f"Character vocabulary size: {len(char_vocab):,}")
print(f"Number of labels: {len(label2id)}")

Word vocabulary size: 21,011
Character vocabulary size: 86
Number of labels: 9


In [45]:
import torch
from torch.utils.data import Dataset, DataLoader

MAX_CHAR_LEN = 20

class NERDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        example = self.data[idx]

        tokens = example["tokens"]
        tags = example["ner_tags"]

        word_ids = [
            word_vocab.get(token.lower(), word_vocab[UNK_TOKEN])
            for token in tokens
        ]

        char_ids = [
            [
                char_vocab.get(char, char_vocab[UNK_TOKEN])
                for char in token[:MAX_CHAR_LEN]
            ]
            for token in tokens
        ]

        return {
            "word_ids": word_ids,
            "char_ids": char_ids,
            "labels": tags
        }

train_dataset = NERDataset(dataset["train"])
val_dataset = NERDataset(dataset["validation"])
test_dataset = NERDataset(dataset["test"])

print("PyTorch datasets created:")
print(f"Train: {len(train_dataset):,}")
print(f"Validation: {len(val_dataset):,}")
print(f"Test: {len(test_dataset):,}")

PyTorch datasets created:
Train: 14,041
Validation: 3,250
Test: 3,453


In [46]:
from torch.nn.utils.rnn import pad_sequence

BATCH_SIZE = 32

def collate_ner_batch(batch):
    word_sequences = [
        torch.tensor(item["word_ids"], dtype=torch.long)
        for item in batch
    ]

    label_sequences = [
        torch.tensor(item["labels"], dtype=torch.long)
        for item in batch
    ]

    word_ids = pad_sequence(
        word_sequences,
        batch_first=True,
        padding_value=word_vocab[PAD_TOKEN]
    )

    labels = pad_sequence(
        label_sequences,
        batch_first=True,
        padding_value=-100
    )

    max_seq_len = word_ids.size(1)

    char_ids = torch.zeros(
        len(batch), max_seq_len, MAX_CHAR_LEN,
        dtype=torch.long
    )

    char_lengths = torch.zeros(
        len(batch), max_seq_len,
        dtype=torch.long
    )

    for i, item in enumerate(batch):
        for j, chars in enumerate(item["char_ids"]):
            char_len = len(chars)
            if char_len > 0:
                char_ids[i, j, :char_len] = torch.tensor(chars)
                char_lengths[i, j] = char_len

    attention_mask = (word_ids != word_vocab[PAD_TOKEN]).long()

    return {
        "word_ids": word_ids,
        "char_ids": char_ids,
        "char_lengths": char_lengths,
        "labels": labels,
        "attention_mask": attention_mask,
        "lengths": attention_mask.sum(dim=1)
    }


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_ner_batch
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_ner_batch
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_ner_batch
)

In [47]:
import torch.nn as nn
import torch.nn.functional as F

class CharacterCNN(nn.Module):
    def __init__(
        self,
        char_vocab_size,
        char_embedding_dim=30,
        num_filters=50,
        kernel_size=3
    ):
        super().__init__()

        self.char_embedding = nn.Embedding(
            char_vocab_size,
            char_embedding_dim,
            padding_idx=0
        )

        self.conv = nn.Conv1d(
            in_channels=char_embedding_dim,
            out_channels=num_filters,
            kernel_size=kernel_size,
            padding=kernel_size // 2
        )

    def forward(self, char_ids):
        batch_size, seq_len, char_len = char_ids.shape

        chars = char_ids.view(-1, char_len)

        embeddings = self.char_embedding(chars)
        embeddings = embeddings.transpose(1, 2)

        features = F.relu(self.conv(embeddings))
        features = F.max_pool1d(features, kernel_size=features.size(2))
        features = features.squeeze(2)

        return features.view(batch_size, seq_len, -1)

In [48]:
class LSTMNER(nn.Module):
    def __init__(
        self,
        vocab_size,
        num_labels,
        embedding_dim=100,
        char_feature_dim=50,
        hidden_dim=128,
        bidirectional=False,
        dropout=0.3
    ):
        super().__init__()

        self.word_embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        self.char_cnn = CharacterCNN(
            char_vocab_size=len(char_vocab),
            num_filters=char_feature_dim
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim + char_feature_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=bidirectional
        )

        output_dim = hidden_dim * (2 if bidirectional else 1)

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(output_dim, num_labels)

    def forward(self, word_ids, char_ids):
        word_features = self.word_embedding(word_ids)
        char_features = self.char_cnn(char_ids)

        features = torch.cat(
            [word_features, char_features],
            dim=-1
        )

        output, _ = self.lstm(features)
        output = self.dropout(output)

        return self.classifier(output)


print("LSTM and BiLSTM architecture is ready.")

LSTM and BiLSTM architecture is ready.


In [49]:
!pip install -q pytorch-crf

In [62]:
class BiLSTM_CRF(nn.Module):
    def __init__(
        self,
        vocab_size,
        num_labels,
        embedding_dim=100,
        char_feature_dim=50,
        hidden_dim=128,
        dropout=0.3
    ):
        super().__init__()

        self.word_embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        self.char_cnn = CharacterCNN(
            len(char_vocab),
            num_filters=char_feature_dim
        )

        self.lstm = nn.LSTM(
            embedding_dim + char_feature_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, num_labels)
        self.crf = CRF(num_labels, batch_first=True)

    def forward(self, word_ids, char_ids, attention_mask, labels=None):
        lengths = attention_mask.sum(dim=1)

        word_features = self.word_embedding(word_ids)
        char_features = self.char_cnn(char_ids)

        features = torch.cat(
            [word_features, char_features], dim=-1
        )

        packed = pack_padded_sequence(
            features,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        packed_output, _ = self.lstm(packed)

        output, _ = pad_packed_sequence(
            packed_output,
            batch_first=True,
            total_length=word_ids.size(1)
        )

        emissions = self.classifier(self.dropout(output))
        mask = attention_mask.bool()

        if labels is not None:
            safe_labels = labels.masked_fill(labels == -100, 0)

            return -self.crf(
                emissions,
                safe_labels,
                mask=mask,
                reduction="mean"
            )

        return self.crf.decode(emissions, mask=mask)


print("Padding-aware BiLSTM + CRF is ready.")

Padding-aware BiLSTM + CRF is ready.


In [51]:
from seqeval.metrics import f1_score, classification_report
from copy import deepcopy
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training device:", device)


def evaluate_model(model, loader, model_type):
    model.eval()

    all_true_labels = []
    all_predictions = []

    with torch.no_grad():
        for batch in loader:
            word_ids = batch["word_ids"].to(device)
            char_ids = batch["char_ids"].to(device)
            labels = batch["labels"].to(device)
            mask = batch["attention_mask"].to(device)

            if model_type == "CRF":
                predictions = model(
                    word_ids, char_ids, mask
                )
            else:
                logits = model(word_ids, char_ids)
                predictions = logits.argmax(dim=-1)

            for i in range(len(labels)):
                seq_len = int(mask[i].sum().item())

                true_sequence = [
                    label_names[tag]
                    for tag in labels[i, :seq_len].tolist()
                ]

                if model_type == "CRF":
                    predicted_sequence = [
                        label_names[tag]
                        for tag in predictions[i]
                    ]
                else:
                    predicted_sequence = [
                        label_names[tag]
                        for tag in predictions[i, :seq_len].tolist()
                    ]

                all_true_labels.append(true_sequence)
                all_predictions.append(predicted_sequence)

    return f1_score(all_true_labels, all_predictions), all_true_labels, all_predictions


def train_model(model, model_type, epochs=10, learning_rate=0.001):
    model = model.to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate
    )

    criterion = nn.CrossEntropyLoss(ignore_index=-100)

    best_f1 = -1
    best_state = None

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

        for batch in progress:
            word_ids = batch["word_ids"].to(device)
            char_ids = batch["char_ids"].to(device)
            labels = batch["labels"].to(device)
            mask = batch["attention_mask"].to(device)

            optimizer.zero_grad()

            if model_type == "CRF":
                loss = model(
                    word_ids, char_ids, mask, labels
                )
            else:
                logits = model(word_ids, char_ids)
                loss = criterion(
                    logits.reshape(-1, logits.size(-1)),
                    labels.reshape(-1)
                )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item()

        avg_train_loss = total_loss / len(train_loader)

        val_f1, _, _ = evaluate_model(
            model, val_loader, model_type
        )

        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Train Loss: {avg_train_loss:.4f} | "
            f"Validation F1: {val_f1:.4f}"
        )

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = deepcopy(model.state_dict())

    model.load_state_dict(best_state)

    print(f"\nBest Validation F1: {best_f1:.4f}")

    return model

Training device: cuda


In [64]:
lstm_model = LSTMNER(
    vocab_size=len(word_vocab),
    num_labels=len(label_names),
    bidirectional=False
)

lstm_model = train_model(
    lstm_model,
    model_type="LSTM",
    epochs=10,
    learning_rate=0.001
)

Epoch 1/10:   0%|          | 0/439 [00:00<?, ?it/s]

Epoch 1/10 | Train Loss: 0.4964 | Validation F1: 0.5011


Epoch 2/10:   0%|          | 0/439 [00:00<?, ?it/s]

Epoch 2/10 | Train Loss: 0.1971 | Validation F1: 0.6725


Epoch 3/10:   0%|          | 0/439 [00:00<?, ?it/s]

Epoch 3/10 | Train Loss: 0.1390 | Validation F1: 0.7032


... (10 outputs left out)

Epoch 9/10:   0%|          | 0/439 [00:00<?, ?it/s]

Epoch 9/10 | Train Loss: 0.0355 | Validation F1: 0.7914


Epoch 10/10:   0%|          | 0/439 [00:00<?, ?it/s]

Epoch 10/10 | Train Loss: 0.0290 | Validation F1: 0.7686

Best Validation F1: 0.7914


In [65]:
bilstm_model = LSTMNER(
    vocab_size=len(word_vocab),
    num_labels=len(label_names),
    bidirectional=True
)

bilstm_model = train_model(
    bilstm_model,
    model_type="BiLSTM",
    epochs=10,
    learning_rate=0.001
)

Epoch 1/10:   0%|          | 0/439 [00:00<?, ?it/s]

Epoch 1/10 | Train Loss: 0.4250 | Validation F1: 0.6462


Epoch 2/10:   0%|          | 0/439 [00:00<?, ?it/s]

Epoch 2/10 | Train Loss: 0.1378 | Validation F1: 0.7546


Epoch 3/10:   0%|          | 0/439 [00:00<?, ?it/s]

Epoch 3/10 | Train Loss: 0.0896 | Validation F1: 0.8034


... (10 outputs left out)

Epoch 9/10:   0%|          | 0/439 [00:00<?, ?it/s]

Epoch 9/10 | Train Loss: 0.0082 | Validation F1: 0.8393


Epoch 10/10:   0%|          | 0/439 [00:00<?, ?it/s]

Epoch 10/10 | Train Loss: 0.0062 | Validation F1: 0.8459

Best Validation F1: 0.8482


In [66]:
bilstm_crf_model = BiLSTM_CRF(
    vocab_size=len(word_vocab),
    num_labels=len(label_names)
)

bilstm_crf_model = train_model(
    bilstm_crf_model,
    model_type="CRF",
    epochs=10,
    learning_rate=0.001
)

Epoch 1/10:   0%|          | 0/439 [00:00<?, ?it/s]

Epoch 1/10 | Train Loss: 5.0744 | Validation F1: 0.7214


Epoch 2/10:   0%|          | 0/439 [00:00<?, ?it/s]

Epoch 2/10 | Train Loss: 1.6557 | Validation F1: 0.8051


Epoch 3/10:   0%|          | 0/439 [00:00<?, ?it/s]

Epoch 3/10 | Train Loss: 1.0141 | Validation F1: 0.8241


... (10 outputs left out)

Epoch 9/10:   0%|          | 0/439 [00:00<?, ?it/s]

Epoch 9/10 | Train Loss: 0.0647 | Validation F1: 0.8675


Epoch 10/10:   0%|          | 0/439 [00:00<?, ?it/s]

Epoch 10/10 | Train Loss: 0.0453 | Validation F1: 0.8668

Best Validation F1: 0.8676


In [56]:
 !pip install -q gensim

import gensim.downloader as api

glove_vectors = api.load("glove-wiki-gigaword-100")

print("GloVe loaded successfully!")
print("Embedding dimension:", glove_vectors.vector_size)

GloVe loaded successfully!
Embedding dimension: 100


In [57]:
import numpy as np

EMBEDDING_DIM = 100

embedding_matrix = np.random.normal(
    0, 0.1,
    (len(word_vocab), EMBEDDING_DIM)
).astype(np.float32)

embedding_matrix[word_vocab[PAD_TOKEN]] = np.zeros(EMBEDDING_DIM)

found_words = 0

for word, idx in word_vocab.items():
    if word in glove_vectors:
        embedding_matrix[idx] = glove_vectors[word]
        found_words += 1

print(f"Embedding matrix shape: {embedding_matrix.shape}")
print(f"Words found in GloVe: {found_words:,}")
print(f"Coverage: {found_words / len(word_vocab) * 100:.2f}%")

Embedding matrix shape: (21011, 100)
Words found in GloVe: 18,415
Coverage: 87.64%


In [63]:
class GloVeBiLSTMNER(nn.Module):
    def __init__(
        self,
        embedding_matrix,
        num_labels,
        char_feature_dim=50,
        hidden_dim=128,
        dropout=0.3
    ):
        super().__init__()

        weights = torch.tensor(
            embedding_matrix,
            dtype=torch.float32
        )

        self.word_embedding = nn.Embedding.from_pretrained(
            weights,
            freeze=False,
            padding_idx=0
        )

        self.char_cnn = CharacterCNN(
            len(char_vocab),
            num_filters=char_feature_dim
        )

        self.lstm = nn.LSTM(
            100 + char_feature_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, num_labels)

    def forward(self, word_ids, char_ids):
        lengths = word_ids.ne(0).sum(dim=1)

        word_features = self.word_embedding(word_ids)
        char_features = self.char_cnn(char_ids)

        features = torch.cat(
            [word_features, char_features], dim=-1
        )

        packed = pack_padded_sequence(
            features,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        packed_output, _ = self.lstm(packed)

        output, _ = pad_packed_sequence(
            packed_output,
            batch_first=True,
            total_length=word_ids.size(1)
        )

        return self.classifier(self.dropout(output))


print("Padding-aware BiLSTM + GloVe is ready.")

Padding-aware BiLSTM + GloVe is ready.


In [67]:
glove_bilstm_model = GloVeBiLSTMNER(
    embedding_matrix=embedding_matrix,
    num_labels=len(label_names)
)

glove_bilstm_model = train_model(
    glove_bilstm_model,
    model_type="BiLSTM",
    epochs=10,
    learning_rate=0.001
)

Epoch 1/10:   0%|          | 0/439 [00:00<?, ?it/s]

Epoch 1/10 | Train Loss: 0.2532 | Validation F1: 0.8419


Epoch 2/10:   0%|          | 0/439 [00:00<?, ?it/s]

Epoch 2/10 | Train Loss: 0.0613 | Validation F1: 0.8829


Epoch 3/10:   0%|          | 0/439 [00:00<?, ?it/s]

Epoch 3/10 | Train Loss: 0.0361 | Validation F1: 0.8834


... (10 outputs left out)

Epoch 9/10:   0%|          | 0/439 [00:00<?, ?it/s]

Epoch 9/10 | Train Loss: 0.0037 | Validation F1: 0.8925


Epoch 10/10:   0%|          | 0/439 [00:00<?, ?it/s]

Epoch 10/10 | Train Loss: 0.0032 | Validation F1: 0.8943

Best Validation F1: 0.8943


In [60]:
!pip install -q transformers accelerate

In [61]:
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

class CharacterCNN(nn.Module):
    def __init__(
        self,
        char_vocab_size,
        char_embedding_dim=30,
        num_filters=50,
        kernel_size=3
    ):
        super().__init__()

        self.char_embedding = nn.Embedding(
            char_vocab_size,
            char_embedding_dim,
            padding_idx=0
        )

        self.conv = nn.Conv1d(
            char_embedding_dim,
            num_filters,
            kernel_size,
            padding=kernel_size // 2
        )

    def forward(self, char_ids):
        batch_size, seq_len, char_len = char_ids.shape

        chars = char_ids.reshape(-1, char_len)
        char_mask = chars.ne(0)

        embeddings = self.char_embedding(chars)
        embeddings = embeddings.transpose(1, 2)

        features = F.relu(self.conv(embeddings))

        features = features.masked_fill(
            ~char_mask.unsqueeze(1),
            float("-inf")
        )

        features = features.max(dim=2).values

        return features.reshape(batch_size, seq_len, -1)


class LSTMNER(nn.Module):
    def __init__(
        self,
        vocab_size,
        num_labels,
        embedding_dim=100,
        char_feature_dim=50,
        hidden_dim=128,
        bidirectional=False,
        dropout=0.3
    ):
        super().__init__()

        self.word_embedding = nn.Embedding(
            vocab_size, embedding_dim, padding_idx=0
        )

        self.char_cnn = CharacterCNN(
            len(char_vocab),
            num_filters=char_feature_dim
        )

        self.lstm = nn.LSTM(
            embedding_dim + char_feature_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=bidirectional
        )

        output_dim = hidden_dim * (2 if bidirectional else 1)

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(output_dim, num_labels)

    def forward(self, word_ids, char_ids):
        lengths = word_ids.ne(0).sum(dim=1)

        word_features = self.word_embedding(word_ids)
        char_features = self.char_cnn(char_ids)

        features = torch.cat(
            [word_features, char_features], dim=-1
        )

        packed = pack_padded_sequence(
            features,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        packed_output, _ = self.lstm(packed)

        output, _ = pad_packed_sequence(
            packed_output,
            batch_first=True,
            total_length=word_ids.size(1)
        )

        return self.classifier(self.dropout(output))


print("Padding-aware Character CNN, LSTM and BiLSTM are ready.")

Padding-aware Character CNN, LSTM and BiLSTM are ready.


In [68]:
!pip install -q transformers accelerate seqeval

In [69]:
from transformers import AutoTokenizer

MODEL_NAME = "bert-base-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=128
    )

    aligned_labels = []

    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)

        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(labels[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        aligned_labels.append(label_ids)

    tokenized_inputs["labels"] = aligned_labels
    return tokenized_inputs


tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names
)

print("BERT tokenization completed!")
print("Train:", len(tokenized_dataset["train"]))
print("Validation:", len(tokenized_dataset["validation"]))
print("Test:", len(tokenized_dataset["test"]))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

BERT tokenization completed!
Train: 14041
Validation: 3250
Test: 3453


In [70]:
from transformers import AutoModelForTokenClassification

bert_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
)

bert_model = bert_model.to(device)

print("BERT model loaded successfully!")
print("Number of labels:", bert_model.config.num_labels)
print("Training device:", device)


model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

BERT model loaded successfully!
Number of labels: 9
Training device: cuda


In [71]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

print("Data collator ready!")


Data collator ready!


In [72]:
import numpy as np
from seqeval.metrics import (
    precision_score,
    recall_score,
    f1_score,
    classification_report
)
from seqeval.scheme import IOB2

def compute_bert_metrics(eval_pred):
    logits, true_labels = eval_pred

    if isinstance(logits, tuple):
        logits = logits[0]

    predictions = np.argmax(logits, axis=2)

    true_sequences = []
    predicted_sequences = []

    for prediction, labels in zip(predictions, true_labels):
        true_sequence = []
        predicted_sequence = []

        for pred_id, label_id in zip(prediction, labels):
            if label_id == -100:
                continue

            true_sequence.append(id2label[int(label_id)])
            predicted_sequence.append(id2label[int(pred_id)])

        true_sequences.append(true_sequence)
        predicted_sequences.append(predicted_sequence)

    report = classification_report(
        true_sequences,
        predicted_sequences,
        mode="strict",
        scheme=IOB2,
        output_dict=True,
        zero_division=0
    )

    metrics = {
        "precision": precision_score(
            true_sequences, predicted_sequences,
            mode="strict", scheme=IOB2, zero_division=0
        ),
        "recall": recall_score(
            true_sequences, predicted_sequences,
            mode="strict", scheme=IOB2, zero_division=0
        ),
        "f1": f1_score(
            true_sequences, predicted_sequences,
            mode="strict", scheme=IOB2, zero_division=0
        )
    }

    for entity in ["PER", "ORG", "LOC", "MISC"]:
        metrics[f"{entity}_f1"] = report.get(entity, {}).get("f1-score", 0.0)

    return metrics


print("Strict IOB2 evaluation metrics are ready!")

Strict IOB2 evaluation metrics are ready!


In [73]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./bert_ner_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=1,
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

print("BERT training configuration ready!")

BERT training configuration ready!


In [74]:
from transformers import Trainer

bert_trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_bert_metrics
)

print("BERT Trainer is ready!")

BERT Trainer is ready!


In [75]:
bert_train_result = bert_trainer.train()

print("\nBERT Fine-tuning completed!")
print("Best checkpoint:", bert_trainer.state.best_model_checkpoint)
print("Best validation F1:", bert_trainer.state.best_metric)

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Per F1,Org F1,Loc F1,Misc F1
1,0.049770,0.041073,0.932005,0.937500,0.934744,0.972707,0.913235,0.952509,0.853007
2,0.029674,0.037744,0.950363,0.948282,0.949321,0.974178,0.926592,0.967865,0.895539
3,0.013992,0.037289,0.949958,0.949798,0.949878,0.975211,0.930493,0.967267,0.893434


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte


BERT Fine-tuning completed!
Best checkpoint: ./bert_ner_results/checkpoint-2634
Best validation F1: 0.9498778535927892


In [76]:
from seqeval.metrics import precision_score, recall_score, f1_score
from seqeval.scheme import IOB2

models_to_compare = {
    "LSTM": (lstm_model, "LSTM"),
    "BiLSTM": (bilstm_model, "BiLSTM"),
    "BiLSTM + CRF": (bilstm_crf_model, "CRF"),
    "BiLSTM + GloVe + Char CNN": (glove_bilstm_model, "BiLSTM")
}

strict_results = {}

for model_name, (model, model_type) in models_to_compare.items():
    _, true_labels, predictions = evaluate_model(
        model, val_loader, model_type
    )

    strict_results[model_name] = {
        "Precision": precision_score(
            true_labels, predictions,
            mode="strict", scheme=IOB2, zero_division=0
        ),
        "Recall": recall_score(
            true_labels, predictions,
            mode="strict", scheme=IOB2, zero_division=0
        ),
        "F1": f1_score(
            true_labels, predictions,
            mode="strict", scheme=IOB2, zero_division=0
        )
    }

bert_validation_results = bert_trainer.evaluate()

strict_results["BERT"] = {
    "Precision": bert_validation_results["eval_precision"],
    "Recall": bert_validation_results["eval_recall"],
    "F1": bert_validation_results["eval_f1"]
}

print("\nSTRICT IOB2 VALIDATION COMPARISON")
print("-" * 65)

for name, metrics in strict_results.items():
    print(
        f"{name:30s} | "
        f"Precision: {metrics['Precision']:.4f} | "
        f"Recall: {metrics['Recall']:.4f} | "
        f"F1: {metrics['F1']:.4f}"
    )

Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Per F1,Org F1,Loc F1,Misc F1
0.013992,0.037289,3,0.949958,0.949798,0.949878,0.975211,0.930493,0.967267,0.893434



STRICT IOB2 VALIDATION COMPARISON
-----------------------------------------------------------------
LSTM                           | Precision: 0.8172 | Recall: 0.8124 | F1: 0.8148
BiLSTM                         | Precision: 0.8634 | Recall: 0.8502 | F1: 0.8568
BiLSTM + CRF                   | Precision: 0.8653 | Recall: 0.8743 | F1: 0.8697
BiLSTM + GloVe + Char CNN      | Precision: 0.9004 | Recall: 0.8992 | F1: 0.8998
BERT                           | Precision: 0.9500 | Recall: 0.9498 | F1: 0.9499


In [77]:
test_results = bert_trainer.predict(tokenized_dataset["test"])

test_metrics = test_results.metrics

print("BERT FINAL TEST RESULTS")
print("-" * 40)

print(f"Precision: {test_metrics['test_precision']:.4f}")
print(f"Recall:    {test_metrics['test_recall']:.4f}")
print(f"F1 Score:  {test_metrics['test_f1']:.4f}")

print("\nPer-Entity F1:")
for entity in ["PER", "ORG", "LOC", "MISC"]:
    key = f"test_{entity}_f1"
    print(f"{entity}: {test_metrics[key]:.4f}")

BERT FINAL TEST RESULTS
----------------------------------------
Precision: 0.9096
Recall:    0.9185
F1 Score:  0.9140

Per-Entity F1:
PER: 0.9626
ORG: 0.8923
LOC: 0.9355
MISC: 0.8062


In [78]:
from seqeval.metrics import classification_report
from seqeval.scheme import IOB2

test_logits = test_results.predictions
test_true_labels = test_results.label_ids

if isinstance(test_logits, tuple):
    test_logits = test_logits[0]

test_predictions = np.argmax(test_logits, axis=2)

test_true_sequences = []
test_pred_sequences = []

for prediction, labels in zip(test_predictions, test_true_labels):
    true_sequence = []
    predicted_sequence = []

    for pred_id, label_id in zip(prediction, labels):
        if label_id == -100:
            continue

        true_sequence.append(id2label[int(label_id)])
        predicted_sequence.append(id2label[int(pred_id)])

    test_true_sequences.append(true_sequence)
    test_pred_sequences.append(predicted_sequence)

print(classification_report(
    test_true_sequences,
    test_pred_sequences,
    mode="strict",
    scheme=IOB2,
    digits=4,
    zero_division=0
))

              precision    recall  f1-score   support

         LOC     0.9352    0.9358    0.9355      1666
        MISC     0.7923    0.8205    0.8062       702
         ORG     0.8795    0.9055    0.8923      1661
         PER     0.9687    0.9567    0.9626      1615

   micro avg     0.9096    0.9185    0.9140      5644
   macro avg     0.8939    0.9046    0.8991      5644
weighted avg     0.9106    0.9185    0.9145      5644



In [81]:
import re
import torch

def predict_entities(text):
    bert_trainer.model.eval()

    # Separate punctuation from words
    words = re.findall(r"\w+(?:['’]\w+)*|[^\w\s]", text)

    encoding = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    word_ids = encoding.word_ids(batch_index=0)

    inputs = {
        key: value.to(device)
        for key, value in encoding.items()
    }

    with torch.no_grad():
        outputs = bert_trainer.model(**inputs)

    predictions = outputs.logits.argmax(dim=-1)[0].cpu().tolist()

    word_labels = {}

    for token_idx, word_idx in enumerate(word_ids):
        if word_idx is not None and word_idx not in word_labels:
            word_labels[word_idx] = id2label[predictions[token_idx]]

    entities = []
    current_entity = None

    for word_idx, word in enumerate(words):
        label = word_labels.get(word_idx, "O")

        if label.startswith("B-"):
            if current_entity:
                entities.append(current_entity)

            current_entity = {
                "text": word,
                "type": label[2:]
            }

        elif (
            label.startswith("I-")
            and current_entity
            and current_entity["type"] == label[2:]
        ):
            current_entity["text"] += " " + word

        else:
            if current_entity:
                entities.append(current_entity)
                current_entity = None

    if current_entity:
        entities.append(current_entity)

    return entities


# Test
sample_text = "Apple announced a new product in London with Tim Cook."

print(predict_entities(sample_text))
visualize_entities(sample_text)

[{'text': 'Apple', 'type': 'ORG'}, {'text': 'London', 'type': 'LOC'}, {'text': 'Tim Cook', 'type': 'PER'}]


In [80]:
from IPython.display import display, HTML
import html

ENTITY_COLORS = {
    "PER": "#DCEBFF",
    "ORG": "#DDF5E3",
    "LOC": "#FFE8CC",
    "MISC": "#EEDFFF"
}

ENTITY_TEXT_COLORS = {
    "PER": "#174EA6",
    "ORG": "#137333",
    "LOC": "#A34B00",
    "MISC": "#7137A8"
}

def visualize_entities(text):
    entities = predict_entities(text)

    highlighted_text = html.escape(text)

    for entity in sorted(entities, key=lambda x: len(x["text"]), reverse=True):
        entity_text = html.escape(entity["text"])
        entity_type = entity["type"]

        highlighted_entity = (
            f'<mark style="background:{ENTITY_COLORS[entity_type]};'
            f'color:{ENTITY_TEXT_COLORS[entity_type]};'
            f'padding:3px 6px;border-radius:5px;font-weight:bold;">'
            f'{entity_text} '
            f'<small>[{entity_type}]</small></mark>'
        )

        highlighted_text = highlighted_text.replace(
            entity_text, highlighted_entity, 1
        )

    display(HTML(
        f'<div style="font-size:17px;line-height:2;'
        f'padding:15px;border:1px solid #ddd;border-radius:10px;">'
        f'{highlighted_text}</div>'
    ))


visualize_entities(
    "Apple announced a new product in London with Tim Cook."
)

In [82]:
!pip -q install flask

In [83]:
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route("/", methods=["GET"])
def home():
    return jsonify({
        "message": "NER API is running!",
        "model": "BERT",
        "status": "ready"
    })


@app.route("/predict", methods=["POST"])
def predict():
    data = request.get_json(silent=True) or {}

    text = data.get("text", "")

    if not isinstance(text, str) or not text.strip():
        return jsonify({
            "error": "Please provide a valid text."
        }), 400

    entities = predict_entities(text)

    return jsonify({
        "input_text": text,
        "entities": entities,
        "total_entities": len(entities)
    })


print("Flask API created successfully!")
print("Available endpoints:")
print("GET  /")
print("POST /predict")

Flask API created successfully!
Available endpoints:
GET  /
POST /predict


In [84]:
from werkzeug.serving import make_server
import threading

flask_server = make_server("0.0.0.0", 5000, app)

flask_thread = threading.Thread(
    target=flask_server.serve_forever,
    daemon=True
)

flask_thread.start()

print("Flask server is running on port 5000!")

Flask server is running on port 5000!


In [88]:
import requests

local_response = requests.get("http://127.0.0.1:5000/")

print("Local Status:", local_response.status_code)
print("Local Response:", local_response.text)
print("Server Thread Alive:", flask_thread.is_alive())

INFO:werkzeug:127.0.0.1 - - [03/Nov/2026 23:47:33] "GET / HTTP/1.1" 200 -


Local Status: 200
Local Response: {"message":"NER API is running!","model":"BERT","status":"ready"}

Server Thread Alive: True


In [89]:
!pip -q install pyngrok

In [90]:
from pyngrok import ngrok
from getpass import getpass

NGROK_AUTH_TOKEN = getpass("Enter your ngrok Auth Token: ")

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

print("ngrok authentication configured successfully!")


Enter your ngrok Auth Token: ··········
ngrok authentication configured successfully!


In [91]:
# Create public tunnel to Flask
public_url = ngrok.connect(5000).public_url

print("NER API Public URL:")
print(public_url)

NER API Public URL:
https://mustard-cahoots-finch.ngrok-free.dev


In [92]:
import requests

api_response = requests.get(
    public_url,
    headers={"ngrok-skip-browser-warning": "true"}
)

print("Status Code:", api_response.status_code)
print("Response:", api_response.json())

INFO:werkzeug:127.0.0.1 - - [03/Nov/2026 23:52:15] "GET / HTTP/1.1" 200 -


Status Code: 200
Response: {'message': 'NER API is running!', 'model': 'BERT', 'status': 'ready'}


In [94]:
import random

random.seed(42)

selected_indices = random.sample(
    range(len(dataset["test"])),
    5
)

selected_examples = []

for idx in selected_indices:
    example = dataset["test"][idx]

    selected_examples.append({
        "tokens": example["tokens"],
        "true_labels": [
            id2label[tag] for tag in example["ner_tags"]
        ]
    })

print("Selected 5 real examples from CoNLL-2003 Test Dataset.")

for i, example in enumerate(selected_examples, 1):
    print(f"\nExample {i}:")
    print(" ".join(example["tokens"]))

Selected 5 real examples from CoNLL-2003 Test Dataset.

Example 1:
( 52.76 / 53.18 )

Example 2:
WESTERN CONFERENCE

Example 3:
Wasim Akram b Harris 4

Example 4:
Mansfield 21 5 9 7 21 22 24

Example 5:
-- New York Commodities Desk , 212-859-1640


In [95]:
import torch

def predict_token_labels(tokens):
    encoding = tokenizer(
        tokens,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    word_ids = encoding.word_ids(batch_index=0)

    inputs = {
        key: value.to(device)
        for key, value in encoding.items()
    }

    bert_trainer.model.eval()

    with torch.no_grad():
        outputs = bert_trainer.model(**inputs)

    predictions = outputs.logits.argmax(dim=-1)[0].cpu().tolist()

    predicted_labels = ["O"] * len(tokens)

    for token_idx, word_idx in enumerate(word_ids):
        if word_idx is not None and predicted_labels[word_idx] == "O":
            predicted_labels[word_idx] = id2label[predictions[token_idx]]

    return predicted_labels


for i, example in enumerate(selected_examples, 1):
    predicted_labels = predict_token_labels(example["tokens"])

    print(f"\n{'='*60}")
    print(f"EXAMPLE {i}")
    print(f"{'='*60}")

    print(f"{'TOKEN':<22} {'GROUND TRUTH':<15} {'BERT PREDICTION'}")
    print("-" * 60)

    for token, true_label, pred_label in zip(
        example["tokens"],
        example["true_labels"],
        predicted_labels
    ):
        if true_label != "O" or pred_label != "O":
            print(f"{token:<22} {true_label:<15} {pred_label}")


EXAMPLE 1
TOKEN                  GROUND TRUTH    BERT PREDICTION
------------------------------------------------------------

EXAMPLE 2
TOKEN                  GROUND TRUTH    BERT PREDICTION
------------------------------------------------------------
WESTERN                O               B-MISC

EXAMPLE 3
TOKEN                  GROUND TRUTH    BERT PREDICTION
------------------------------------------------------------
Wasim                  B-PER           B-PER
Akram                  I-PER           I-PER
Harris                 B-PER           B-PER

EXAMPLE 4
TOKEN                  GROUND TRUTH    BERT PREDICTION
------------------------------------------------------------
Mansfield              B-ORG           B-ORG

EXAMPLE 5
TOKEN                  GROUND TRUTH    BERT PREDICTION
------------------------------------------------------------
New                    B-ORG           B-ORG
York                   I-ORG           I-ORG
Commodities            I-ORG           I-ORG
Desk

In [96]:
from seqeval.metrics import precision_score, recall_score, f1_score
from seqeval.scheme import IOB2

all_true = []
all_pred = []

for example in selected_examples:
    predicted_labels = predict_token_labels(example["tokens"])

    all_true.append(example["true_labels"])
    all_pred.append(predicted_labels)

print("Dataset Explorer Results")
print("-" * 40)

print(f"Precision: {precision_score(all_true, all_pred, mode='strict', scheme=IOB2):.4f}")
print(f"Recall:    {recall_score(all_true, all_pred, mode='strict', scheme=IOB2):.4f}")
print(f"F1-score:  {f1_score(all_true, all_pred, mode='strict', scheme=IOB2):.4f}")

Dataset Explorer Results
----------------------------------------
Precision: 0.6000
Recall:    0.7500
F1-score:  0.6667


In [97]:
from seqeval.metrics.sequence_labeling import get_entities

for i, example in enumerate(selected_examples, 1):
    predicted_labels = predict_token_labels(example["tokens"])

    true_entities = get_entities(example["true_labels"])
    predicted_entities = get_entities(predicted_labels)

    print(f"\nEXAMPLE {i}")
    print("-" * 60)

    print("Ground Truth Entities:")
    for entity_type, start, end in true_entities:
        print(" ".join(example["tokens"][start:end+1]), entity_type)

    print("\nBERT Predicted Entities:")
    for entity_type, start, end in predicted_entities:
        print(" ".join(example["tokens"][start:end+1]), entity_type)


EXAMPLE 1
------------------------------------------------------------
Ground Truth Entities:

BERT Predicted Entities:

EXAMPLE 2
------------------------------------------------------------
Ground Truth Entities:

BERT Predicted Entities:
WESTERN MISC

EXAMPLE 3
------------------------------------------------------------
Ground Truth Entities:
Wasim Akram PER
Harris PER

BERT Predicted Entities:
Wasim Akram PER
Harris PER

EXAMPLE 4
------------------------------------------------------------
Ground Truth Entities:
Mansfield ORG

BERT Predicted Entities:
Mansfield ORG

EXAMPLE 5
------------------------------------------------------------
Ground Truth Entities:
New York Commodities ORG

BERT Predicted Entities:
New York Commodities Desk ORG


In [98]:
!pip -q install gradio

In [102]:
import gradio as gr
import html

COLORS = {
    "PER": ("#DBEAFE", "#1D4ED8"),
    "ORG": ("#DCFCE7", "#15803D"),
    "LOC": ("#FFEDD5", "#C2410C"),
    "MISC": ("#F3E8FF", "#7E22CE")
}

def compact_ner(text):
    if not text or not text.strip():
        return "<p>Please enter text.</p>", []

    entities = predict_entities(text)
    highlighted = html.escape(text)

    for entity in sorted(entities, key=lambda x: len(x["text"]), reverse=True):
        word = html.escape(entity["text"])
        label = entity["type"]
        bg, fg = COLORS[label]

        tag = (
            f'<span style="background:{bg};color:{fg};'
            f'padding:4px 7px;border-radius:6px;font-weight:700;">'
            f'{word} [{label}]</span>'
        )

        highlighted = highlighted.replace(word, tag, 1)

    result = f"""
    <div style="font-size:17px;line-height:2.3;padding:14px;
    background:#ffffff;border-radius:10px;color:#172033;">
        {highlighted}
    </div>
    """

    rows = [
        [entity["text"], entity["type"]]
        for entity in entities
    ]

    return result, rows


css = """
.gradio-container {
    max-width: 850px !important;
    margin: auto !important;
}

#header {
    text-align: center;
    background: linear-gradient(120deg, #111827, #312e81, #6d28d9);
    color: #ffffff !important;
    padding: 20px;
    border-radius: 15px;
    margin-bottom: 15px;
}

#header h1 {
    color: #ffffff !important;
    font-size: 30px !important;
    font-weight: 800 !important;
    margin: 0 0 8px 0;
}

#header p {
    color: #f1f5f9 !important;
    font-size: 16px !important;
    margin: 5px 0;
}

#header small {
    color: #ddd6fe !important;
    font-size: 13px !important;
}

"""

with gr.Blocks(
    title="NER Intelligence Studio",
    theme=gr.themes.Soft(
        primary_hue="violet",
        secondary_hue="blue",
        neutral_hue="slate"
    ),
    css=css
) as demo:

    gr.HTML("""
    <div id="header">
        <h1>NER Intelligence Studio</h1>
        <p>AI-Powered Named Entity Recognition</p>
        <small>BERT • CoNLL-2003 • Deep Learning</small>
    </div>
    """)

    gr.Markdown("### Enter Your Text")

    text_input = gr.Textbox(
        placeholder="Apple announced a new product in London with Tim Cook.",
        lines=2,
        label="Input Text"
    )

    analyze = gr.Button(
        "✨ Analyze Text",
        variant="primary"
    )

    gr.Markdown("### 🎨 Detected Entities")

    highlighted = gr.HTML()

    table = gr.Dataframe(
        headers=["Entity", "Type"],
        datatype=["str", "str"],
        interactive=False,
        label="Extracted Entities"
    )

    analyze.click(
        fn=compact_ner,
        inputs=text_input,
        outputs=[highlighted, table]
    )

    gr.Examples(
        examples=[
            ["Apple announced a new product in London with Tim Cook."],
            ["Wasim Akram played cricket for Pakistan."]
        ],
        inputs=text_input,
        label="Try an Example"
    )

demo.launch(
    share=True
)

/tmp/ipykernel_725/1239846917.py:81: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9845cbebbc1358f970.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [104]:
from seqeval.metrics.sequence_labeling import get_entities
import random

random.seed(42)

examples_with_entities = []

for i, example in enumerate(dataset["test"]):
    tokens = example["tokens"]
    labels = [id2label[x] for x in example["ner_tags"]]

    entities = get_entities(labels)

    if entities and 6 <= len(tokens) <= 25:
        examples_with_entities.append({
            "index": i,
            "sentence": " ".join(tokens),
            "entities": entities
        })

# Select 15 different examples
selected = random.sample(
    examples_with_entities,
    min(15, len(examples_with_entities))
)

for number, example in enumerate(selected, 1):
    print(f"\n{'='*55}")
    print(f"Example {number} | Dataset Index: {example['index']}")
    print("Sentence:", example["sentence"])
    print("Ground Truth:", example["entities"])

print(f"\nTotal suitable examples found: {len(examples_with_entities)}")


Example 1 | Dataset Index: 2934
Sentence: Ross County 17 8 3 6 27 23 27
Ground Truth: [('ORG', 0, 1)]

Example 2 | Dataset Index: 431
Sentence: NBA BASKETBALL - STANDINGS AFTER THURSDAY 'S GAMES .
Ground Truth: [('ORG', 0, 0)]

Example 3 | Dataset Index: 79
Sentence: Japan - Hassan Abbas 84 own goal , Takuya Takagi 88 .
Ground Truth: [('LOC', 0, 0), ('PER', 2, 3), ('PER', 8, 9)]

Example 4 | Dataset Index: 3281
Sentence: DALLAS 6 11 .353 9 1/2
Ground Truth: [('ORG', 0, 0)]

Example 5 | Dataset Index: 1220
Sentence: Although not threatening Mauritius directly , it is coming closer to the island and could change direction , it added .
Ground Truth: [('LOC', 3, 3)]

Example 6 | Dataset Index: 1057
Sentence: " He ( Gorst ) isreserving the right not to cooperate , but he has not resigned the whip .
Ground Truth: [('PER', 3, 3)]

Example 7 | Dataset Index: 912
Sentence: * The UDMR wants to contribute to social reform and economic revival in Romania , union leader Marko Bela said .
Ground Tr